# 03.1.1 - RAG Hybrid + Query Rewriting (Few-Shot ala Paper Ma dkk. 2023)

Notebook ini adalah **versi baru** dari `03.1 QR - OpenAI.ipynb` dengan satu perubahan utama: prompt untuk Query Rewriter diadaptasi agar **persis mengikuti paper** Ma, Gong, He, Zhao, Duan (2023) *Query Rewriting for Retrieval-Augmented Large Language Models* (EMNLP 2023, arXiv:2305.14283).

## Perbedaan dengan versi lama (03.1)

| Aspek | 03.1 (lama) | 03.1.1 (baru) |
|---|---|---|
| Prompt style | Zero-shot domain-specific | **Few-shot dengan 3 contoh medical** (mengikuti Tabel 1 paper) |
| Instruksi | `Rewrite the following medical question...` | **`Think step by step` + provide search engine queries** (verbatim paper) |
| Output format | Satu kueri natural language | **Multi-kueri dipisah `;`, diakhiri `**`** (mengikuti paper) |
| Retrieval | Satu kueri -> BM25 + Dense | **Multi-kueri -> BM25 + Dense per kueri -> RRF gabungan** |

## Pipeline
```
original query
  -> Few-shot LLM rewriter (GPT-4.1-mini, temp=0.3)
  -> 1..n rewritten queries (split by ';')
  -> untuk tiap query: BM25 top-50 + Dense top-50
  -> gabung semua rank list via RRF (k=60)
  -> top-5
  -> OpenAI generate (pakai original query)
```

## Hipotesis
Few-shot prompting + multi-query rewriting akan **mengurangi semantic drift** yang ditemukan pada eksperimen 03.1 (akurasi turun -1,2pp dibanding baseline) dan **mendekati pola peningkatan paper** (+1,9pp rata-rata pada open-domain QA).

Hasil disimpan terpisah di `results/qr_v2_openai_phase{1,2}_*.json` agar tidak menimpa hasil lama.

In [1]:
# Install dependencies (jalankan sekali saja)
# !pip install openai rank-bm25 chromadb datasets

In [2]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime

from openai import OpenAI
from rank_bm25 import BM25Okapi
from datasets import load_dataset
import chromadb

warnings.filterwarnings('ignore')
print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | chromadb: {chromadb.__version__}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5 | chromadb: 1.5.8


In [3]:
# ============================================================
# KONFIGURASI
# ============================================================
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'REDACTED_SET_VIA_ENV')

LLM_MODEL   = 'gpt-4.1-mini'
EMBED_MODEL = 'text-embedding-3-small'  # 1536 dim

TOP_K_BM25      = 50   # BM25 ambil top-50 per kueri
TOP_K_DENSE     = 50   # Dense ambil top-50 per kueri
TOP_K_RETRIEVAL = 5    # Final top-5 setelah RRF
MAX_REWRITE_QUERIES = 3  # batas atas jumlah kueri hasil rewrite (paper: 1..n)

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'
MAX_SAMPLES    = 500

TEMPERATURE = 0.0
SEED        = 42

NOTEBOOK_DIR    = Path('.')
BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'
CHROMA_DB_PATH  = NOTEBOOK_DIR / 'pubmedqa_chroma'
RESULTS_DIR     = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

# Hasil disimpan terpisah dari run 03.1 lama supaya bisa dibandingkan
CONFIG_NAME        = 'qr_v2_openai'
PHASE1_PATH        = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'
PHASE2_CUSTOM_PATH = RESULTS_DIR / f'{CONFIG_NAME}_phase2_custom.json'

print('Konfigurasi (Few-Shot QR ala Paper Ma dkk. 2023):')
print(f'  LLM            : {LLM_MODEL} (via OpenAI)')
print(f'  Embedder       : {EMBED_MODEL}')
print(f'  Retriever      : BM25 top-{TOP_K_BM25} + Dense top-{TOP_K_DENSE} -> RRF -> top-{TOP_K_RETRIEVAL}')
print(f'  Max rewrite Q  : {MAX_REWRITE_QUERIES} (multi-query supported)')
print(f'  Vector DB      : chromadb (path: {CHROMA_DB_PATH})')
print(f'  Sampel         : {MAX_SAMPLES}')
print(f'  Config         : {CONFIG_NAME}')
print()
if 'YOUR_API_KEY' in OPENAI_API_KEY or 'YOUR_OPENAI_KEY' in OPENAI_API_KEY:
    print('  OPENAI_API_KEY belum diisi! Set env var OPENAI_API_KEY atau isi di cell ini.')
else:
    print(f'  OPENAI_API_KEY: {OPENAI_API_KEY[:8]}...{OPENAI_API_KEY[-4:]}')


Konfigurasi (Few-Shot QR ala Paper Ma dkk. 2023):
  LLM            : gpt-4.1-mini (via OpenAI)
  Embedder       : text-embedding-3-small
  Retriever      : BM25 top-50 + Dense top-50 -> RRF -> top-5
  Max rewrite Q  : 3 (multi-query supported)
  Vector DB      : chromadb (path: pubmedqa_chroma)
  Sampel         : 500
  Config         : qr_v2_openai

  OPENAI_API_KEY: sk-proj-...FXQA


In [4]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str


@dataclass
class RetrievalResult:
    document       : Document
    score          : float           # BM25 score (atau RRF score)
    doc_id         : int = -1        # index di documents list
    bm25_score     : float = 0.0
    dense_score    : float = 0.0
    rrf_score      : float = 0.0
    reranker_score : float = 0.0


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


In [5]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index baru...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


_full_data            = load_dataset(DATASET_NAME, DATASET_SUBSET, trust_remote_code=True)['train']
bm25_index, documents = load_or_build_bm25(_full_data.select(range(500)))
pubmedqa_data         = _full_data.select(range(MAX_SAMPLES))
print(f'\nEvaluasi akan menggunakan {len(pubmedqa_data)} sampel pertama.')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat BM25 index dari pubmedqa_bm25.pkl...
Dimuat: 1706 dokumen

Evaluasi akan menggunakan 500 sampel pertama.


In [6]:
# ============================================================
# Setup OpenAI Client
# ============================================================
openai_client = OpenAI(api_key=OPENAI_API_KEY)


def openai_generate(prompt: str, max_tokens: int = 300, temperature: float = TEMPERATURE) -> str:
    """Wrapper OpenAI chat completion dengan retry."""
    for attempt in range(5):
        try:
            response = openai_client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
                seed=SEED,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = (attempt + 1) * 10
                print(f'  [Rate limit] Tunggu {wait}s...')
                time.sleep(wait)
            elif '500' in err or '502' in err or '503' in err:
                wait = (attempt + 1) * 5
                print(f'  [Server error] Tunggu {wait}s...')
                time.sleep(wait)
            else:
                print(f'  [OpenAI Error] {type(e).__name__}: {err[:100]}')
                raise
    raise RuntimeError('OpenAI API gagal setelah 5 percobaan.')


def openai_embed(texts: List[str], model: str = EMBED_MODEL) -> List[List[float]]:
    """Wrapper OpenAI embeddings dengan retry. Accepts list of texts, returns list of vectors."""
    for attempt in range(5):
        try:
            response = openai_client.embeddings.create(model=model, input=texts)
            return [d.embedding for d in response.data]
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = (attempt + 1) * 10
                print(f'  [Rate limit] Tunggu {wait}s...')
                time.sleep(wait)
            elif '500' in err or '502' in err or '503' in err:
                wait = (attempt + 1) * 5
                print(f'  [Server error] Tunggu {wait}s...')
                time.sleep(wait)
            else:
                print(f'  [Embed Error] {type(e).__name__}: {err[:100]}')
                raise
    raise RuntimeError('OpenAI embeddings gagal setelah 5 percobaan.')


# Smoke test
print('Testing OpenAI API...')
_test = openai_generate('Reply with exactly: OK', max_tokens=5)
print(f'  Chat response: {_test!r}')
_emb = openai_embed(['aspirin reduces heart attack risk'])
print(f'  Embed dim: {len(_emb[0])} (expected: 1536 for text-embedding-3-small)')
print('OpenAI client siap!')

Testing OpenAI API...
  Chat response: 'OK'
  Embed dim: 1536 (expected: 1536 for text-embedding-3-small)
OpenAI client siap!


In [7]:
# ============================================================
# Build / Load ChromaDB index dengan OpenAI embeddings
# Embed sekali, di-cache di CHROMA_DB_PATH
# ============================================================

def build_or_load_chroma_index(documents: List[Document]) -> chromadb.Collection:
    """
    Build ChromaDB persistent collection dengan OpenAI embeddings.
    Kalau sudah ada dan count-nya match, load saja.
    """
    chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
    collection_name = 'pubmedqa_docs'

    try:
        collection = chroma_client.get_collection(name=collection_name)
        count = collection.count()
        if count == len(documents):
            print(f'Load Chroma collection "{collection_name}" ({count} dokumen) dari {CHROMA_DB_PATH}')
            return collection
        else:
            print(f'Count mismatch: chroma={count}, documents={len(documents)}. Rebuild...')
            chroma_client.delete_collection(name=collection_name)
    except Exception:
        pass

    print(f'Build Chroma collection ({len(documents)} dokumen)...')
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={'hnsw:space': 'cosine'}
    )

    # Batch embedding
    BATCH = 100
    t0 = time.time()
    for start in range(0, len(documents), BATCH):
        batch_docs = documents[start:start + BATCH]
        batch_texts = [d.text[:8000] for d in batch_docs]  # truncate untuk safety
        batch_ids   = [str(start + i) for i in range(len(batch_docs))]
        batch_meta  = [{'pubid': d.pubid, 'section': d.section_label} for d in batch_docs]

        embeddings = openai_embed(batch_texts)
        collection.add(
            ids=batch_ids,
            documents=batch_texts,
            metadatas=batch_meta,
            embeddings=embeddings,
        )
        done = start + len(batch_docs)
        eta  = (time.time()-t0)/done*(len(documents)-done)/60 if done < len(documents) else 0
        print(f'  [{done}/{len(documents)}] embedded | ETA {eta:.1f} mnt')

    print(f'Chroma index dibangun dalam {(time.time()-t0)/60:.1f} menit')
    return collection


chroma_collection = build_or_load_chroma_index(documents)
print(f'\nChroma index siap: {chroma_collection.count()} dokumen')

Load Chroma collection "pubmedqa_docs" (1706 dokumen) dari pubmedqa_chroma

Chroma index siap: 1706 dokumen


In [8]:
# ============================================================
# Query Rewriting - Few-Shot ala Paper Ma dkk. (2023)
# ============================================================
# Format prompt mengikuti Tabel 1 paper:
#   - Instruksi 'Think step by step ...'
#   - Output: queries dipisah ';' diakhiri '**'
#   - 1..n rewritten queries (multi-query untuk dekomposisi)
#   - 3 demonstration examples khusus domain biomedis (PubMedQA)

QUERY_REWRITE_PROMPT = (
    "Think step by step to answer this question, and provide search engine "
    "queries for knowledge that you need. Split the queries with ';' and "
    "end the queries with '**'.\n\n"
    # Demonstration 1: ekspansi singkatan medis
    "Question: Does aspirin reduce the risk of MI in patients with diabetes?\n"
    "Answer: aspirin myocardial infarction prevention diabetes; "
    "low-dose aspirin cardiovascular risk type 2 diabetes patients **\n\n"
    # Demonstration 2: dekomposisi pertanyaan multi-aspek
    "Question: Is metformin effective for T2DM in elderly patients with renal impairment?\n"
    "Answer: metformin type 2 diabetes mellitus efficacy elderly; "
    "metformin renal impairment safety; "
    "metformin contraindications kidney function older adults **\n\n"
    # Demonstration 3: parafrasa istilah umum -> istilah klinis
    "Question: Can regular exercise lower blood pressure in older adults?\n"
    "Answer: physical exercise hypertension blood pressure reduction elderly; "
    "aerobic exercise antihypertensive effect older adults **\n\n"
    # Pertanyaan baru
    "Question: {query}\n"
    "Answer:"
)


def parse_rewriter_output(text: str, max_queries: int = MAX_REWRITE_QUERIES) -> list:
    """Parse output rewriter ala paper: '<q1>; <q2>; ... **'."""
    if not text:
        return []
    # Buang segala sesudah '**' (end marker paper)
    if '**' in text:
        text = text.split('**', 1)[0]
    # Buang prefix 'Answer:' kalau model meniru itu
    text = re.sub(r'^\s*Answer\s*:\s*', '', text, flags=re.IGNORECASE)
    # Pisah dengan ';'
    queries = [q.strip() for q in text.split(';')]
    # Buang query kosong / terlalu pendek
    queries = [q for q in queries if len(q) >= 3]
    # Batasi jumlah
    return queries[:max_queries]


def rewrite_query(query: str) -> list:
    """Reformulasi query (few-shot, multi-query). Return list of queries."""
    prompt = QUERY_REWRITE_PROMPT.format(query=query)
    try:
        raw = openai_generate(prompt, max_tokens=200, temperature=0.3)
        queries = parse_rewriter_output(raw)
        return queries if queries else [query]
    except Exception as e:
        print(f'  [QR Error] {e} -- pakai query asli')
        return [query]


# Test QR
print('Contoh Query Rewriting (few-shot, multi-query):')
for q in ['Does aspirin reduce the risk of MI?',
          'Can exercise prevent T2DM?',
          'Is statin therapy beneficial for stroke prevention in patients over 75?']:
    rws = rewrite_query(q)
    print(f'\nAsli      : {q}')
    for i, rw in enumerate(rws, 1):
        print(f'Rewrite {i} : {rw}')


Contoh Query Rewriting (few-shot, multi-query):

Asli      : Does aspirin reduce the risk of MI?
Rewrite 1 : aspirin myocardial infarction prevention
Rewrite 2 : aspirin cardiovascular risk reduction
Rewrite 3 : aspirin primary secondary prevention MI

Asli      : Can exercise prevent T2DM?
Rewrite 1 : exercise prevention type 2 diabetes mellitus
Rewrite 2 : physical activity risk reduction type 2 diabetes
Rewrite 3 : exercise impact glucose metabolism diabetes prevention

Asli      : Is statin therapy beneficial for stroke prevention in patients over 75?
Rewrite 1 : statin therapy stroke prevention elderly over 75
Rewrite 2 : statins cardiovascular risk reduction older adults
Rewrite 3 : lipid-lowering therapy stroke risk age above 75


In [9]:
# ============================================================
# Dense retrieval + Multi-Query RRF fusion
# ============================================================

def retrieve_dense(query: str, k: int = 50) -> List[Tuple[int, float]]:
    """Dense retrieval via ChromaDB."""
    qvec = openai_embed([query])[0]
    results = chroma_collection.query(
        query_embeddings=[qvec],
        n_results=k,
        include=['distances']
    )
    doc_ids   = [int(i) for i in results['ids'][0]]
    distances = results['distances'][0]
    scores    = [1.0 - d for d in distances]
    return list(zip(doc_ids, scores))


def retrieve_bm25_raw(query: str, k: int = 50) -> List[Tuple[int, float]]:
    """BM25 retrieval."""
    tokens = tokenize_bm25(query)
    scores = bm25_index.get_scores(tokens)
    top_k  = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_k]


def reciprocal_rank_fusion(
    rank_lists: List[List[int]],
    k: int = 60
) -> List[Tuple[int, float]]:
    """Reciprocal Rank Fusion. Gabung beberapa rank list jadi satu."""
    scores = {}
    for rank_list in rank_lists:
        for rank, doc_id in enumerate(rank_list):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


def retrieve_hybrid(
    query: str,
    k_bm25: int = TOP_K_BM25,
    k_dense: int = TOP_K_DENSE,
    k_final: int = TOP_K_RETRIEVAL
) -> Tuple[List[RetrievalResult], List[str]]:
    """
    Hybrid retrieval dengan MULTI-QUERY rewriting (ala paper Ma dkk. 2023).

    Untuk tiap rewritten query, jalankan BM25 + Dense paralel.
    Semua rank list (2 * num_queries) digabung dengan satu RRF.
    """
    # Multi-query rewriting
    rewritten_queries = rewrite_query(query)

    # Kumpulkan rank list dari setiap kueri (BM25 dan Dense terpisah)
    all_rank_lists = []
    bm25_score_lookup  = {}  # ambil skor BM25 tertinggi antar kueri
    dense_score_lookup = {}  # ambil skor dense tertinggi antar kueri

    for rq in rewritten_queries:
        bm25_results  = retrieve_bm25_raw(rq, k=k_bm25)
        dense_results = retrieve_dense(rq, k=k_dense)

        all_rank_lists.append([doc_id for doc_id, _ in bm25_results])
        all_rank_lists.append([doc_id for doc_id, _ in dense_results])

        for doc_id, score in bm25_results:
            bm25_score_lookup[doc_id] = max(bm25_score_lookup.get(doc_id, 0.0), score)
        for doc_id, score in dense_results:
            dense_score_lookup[doc_id] = max(dense_score_lookup.get(doc_id, 0.0), score)

    # RRF fusion seluruh rank list (2*N)
    fused = reciprocal_rank_fusion(all_rank_lists, k=60)

    # Build top-K hasil akhir
    results = []
    for doc_id, rrf_score in fused[:k_final]:
        results.append(RetrievalResult(
            document=documents[doc_id],
            score=rrf_score,
            doc_id=doc_id,
            bm25_score=bm25_score_lookup.get(doc_id, 0.0),
            dense_score=dense_score_lookup.get(doc_id, 0.0),
            rrf_score=rrf_score,
        ))
    return results, rewritten_queries


# Test retrieval
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r, test_rws = retrieve_hybrid(test_q)
print(f'Query asli   : {test_q}')
print(f'Rewrite ({len(test_rws)} kueri):')
for i, rw in enumerate(test_rws, 1):
    print(f'  [{i}] {rw}')
print(f'\nTop-{TOP_K_RETRIEVAL} dokumen (Hybrid + Multi-Query RRF):')
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] RRF={r.rrf_score:.4f} | BM25={r.bm25_score:.2f} | Dense={r.dense_score:.3f} '
          f'| {r.document.section_label} | {r.document.text[:70]}...')


Query asli   : Does aspirin reduce the risk of myocardial infarction?
Rewrite (3 kueri):
  [1] aspirin myocardial infarction prevention
  [2] aspirin cardiovascular risk reduction
  [3] aspirin primary secondary prevention MI

Top-5 dokumen (Hybrid + Multi-Query RRF):
  [1] RRF=0.0769 | BM25=13.09 | Dense=0.473 | BACKGROUND AND PURPOSE | In primary and secondary prevention trials, statins have been shown to...
  [2] RRF=0.0704 | BM25=5.99 | Dense=0.460 | METHODS | This randomized single-blinded single-center clinical trial involved 3...
  [3] RRF=0.0699 | BM25=6.27 | Dense=0.475 | OBJECTIVE | Myocardial damage that is associated with percutaneous coronary interv...
  [4] RRF=0.0697 | BM25=9.52 | Dense=0.436 | METHODS | Our population included 3011 patients without any cancer diagnosis who...
  [5] RRF=0.0586 | BM25=5.98 | Dense=0.458 | BACKGROUND | Although consensus guidelines for pretreatment evaluation and monitori...


In [10]:
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100%% certain.\n\n'
    'Answer:'
)


def generate_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    """Generate jawaban via OpenAI pakai original query."""
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    return openai_generate(
        GENERATION_PROMPT.format(context=context, question=query),
        max_tokens=300,
        temperature=TEMPERATURE
    )


# Test
test_ans = generate_answer(test_q, test_r)
print('Output generation:')
print('-' * 60)
print(test_ans)
print('-' * 60)

Output generation:
------------------------------------------------------------
The provided abstracts discuss the effects of statins on stroke risk, remote postischemic conditioning on myocardial damage during PCI, and bezafibrate on cancer incidence, but none mention aspirin or its impact on myocardial infarction risk. Therefore, there is no relevant information to determine whether aspirin reduces the risk of myocardial infarction.

maybe
------------------------------------------------------------


In [11]:
def extract_label(answer: str) -> str:
    """Ekstrak prediksi yes/no/maybe dari teks jawaban."""
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


cases = [
    ('Strong evidence.\nyes', 'yes'),
    ('No effect found.\nno',  'no'),
    ('Mixed results.\nmaybe', 'maybe'),
    ('Verdict: yes.',          'yes'),
    ('Totally unclear.',       'maybe'),
]
all_ok = all(extract_label(txt) == exp for txt, exp in cases)
print(f'Unit test extract_label: {"PASS" if all_ok else "FAIL"}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label: PASS
Label dari test answer: 'maybe'


In [12]:
# ============================================================
# Custom Zero-NaN Evaluator — 4 metrik via OpenAI
# ============================================================

def _split_sentences(text: str) -> List[str]:
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]


def _llm_yes_no(prompt: str) -> bool:
    try:
        resp = openai_generate(prompt, max_tokens=10, temperature=0.0)
        return 'yes' in resp.lower()[:15]
    except Exception:
        return False


def compute_faithfulness(answer: str, contexts: List[str]) -> float:
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:1500]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement directly supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    supported = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return supported / len(sentences)


def compute_context_recall(reference: str, contexts: List[str]) -> float:
    sentences = _split_sentences(reference)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:1500]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    covered = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return covered / len(sentences)


def compute_answer_relevancy(question: str, answer: str) -> float:
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    prompt_tmpl = (
        'Question: {question}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement relevant to answering the question above? '
        'Answer with only "yes" or "no".'
    )
    relevant = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(question=question, sent=s)))
    return relevant / len(sentences)


def compute_context_precision(question: str, contexts: List[str], reference: str) -> float:
    if not contexts:
        return 0.0
    prompt_tmpl = (
        'Question: {question}\n\n'
        'Ground truth answer: {reference}\n\n'
        'Retrieved context: {ctx}\n\n'
        'Does this context contain information useful for correctly answering '
        'the question based on the ground truth? Answer with only "yes" or "no".'
    )
    relevance = []
    for ctx in contexts:
        is_rel = _llm_yes_no(prompt_tmpl.format(
            question=question, reference=reference[:800], ctx=ctx[:400]
        ))
        relevance.append(1 if is_rel else 0)
    total_relevant = sum(relevance)
    if total_relevant == 0:
        return 0.0
    precision_sum = 0.0
    relevant_count = 0
    for k, rel in enumerate(relevance):
        if rel:
            relevant_count += 1
            precision_sum += relevant_count / (k + 1)
    return precision_sum / total_relevant


def evaluate_custom(question: str, answer: str,
                    contexts: List[str], reference: str) -> Dict:
    return {
        'faithfulness'      : compute_faithfulness(answer, contexts),
        'context_recall'    : compute_context_recall(reference, contexts),
        'answer_relevancy'  : compute_answer_relevancy(question, answer),
        'context_precision' : compute_context_precision(question, contexts, reference),
    }


# Smoke test
_ctx = ['Aspirin reduces blood clotting and is used for heart attack prevention.']
_ans = 'Aspirin helps prevent heart attacks. It works by reducing clotting.'
_ref = 'Aspirin is used for heart attack prevention by reducing blood clotting.'
_r   = evaluate_custom('Does aspirin prevent heart attacks?', _ans, _ctx, _ref)
print('Smoke test (4 metrik):')
for k, v in _r.items():
    print(f'  {k} = {v:.3f}')
print('Zero-NaN evaluator siap.')

Smoke test (4 metrik):
  faithfulness = 1.000
  context_recall = 1.000
  answer_relevancy = 1.000
  context_precision = 1.000
Zero-NaN evaluator siap.


## Demo — 5 Sampel Pertama

In [ ]:
DEMO_SIZE = 5

print(f'DEMO: {DEMO_SIZE} sampel pertama (Hybrid + Few-Shot Multi-Query QR, {LLM_MODEL})')
print('=' * 70)

for i in range(DEMO_SIZE):
    s          = pubmedqa_data[i]
    q, gt, ref = s['question'], s['final_decision'], s['long_answer']
    retrieved, rewritten_qs = retrieve_hybrid(q)
    answer    = generate_answer(q, retrieved)
    predicted = extract_label(answer)
    mark = 'OK' if predicted == gt else 'X '
    print(f'\n[{i+1}] {mark} pred={predicted}, gt={gt}')
    print(f'    Q : {q[:80]}')
    for j, rq in enumerate(rewritten_qs, 1):
        print(f'    rw{j}: {rq[:80]}')


## Phase 1 — Generate Jawaban (500 Sampel)

Estimasi waktu: ~15-20 menit dengan GPT-4.1-mini. Resume otomatis.

In [13]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Memulai Fase 1: {MAX_SAMPLES} sampel (Hybrid + Few-Shot Multi-Query QR).')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()

    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']

        retrieved, rewritten_qs = retrieve_hybrid(q)
        answer    = generate_answer(q, retrieved)
        predicted = extract_label(answer)

        phase1_results.append({
            'idx'              : i,
            'pubid'            : str(s['pubid']),
            'question'         : q,
            'rewritten_queries': rewritten_qs,    # list, bukan single string
            'num_rewrites'     : len(rewritten_qs),
            'ground_truth'     : gt,
            'predicted_label'  : predicted,
            'is_correct'       : predicted == gt,
            'answer'           : answer,
            'contexts'         : [r.document.text for r in retrieved],
            'reference'        : ref,
            'retrieval_scores' : [r.bm25_score  for r in retrieved],
            'dense_scores'     : [r.dense_score for r in retrieved],
            'rrf_scores'       : [r.rrf_score   for r in retrieved],
        })

        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({'config': CONFIG_NAME,
                           'llm_model': LLM_MODEL,
                           'embed_model': EMBED_MODEL,
                           'qr_style': 'few-shot multi-query (Ma dkk. 2023)',
                           'timestamp': datetime.now().isoformat(),
                           'max_samples': MAX_SAMPLES, 'completed': i+1,
                           'results': phase1_results}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            avg_rw = sum(r['num_rewrites'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Akurasi: {acc:.1%} | avg rewrites: {avg_rw:.1f} | '
                  f'pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')

    print(f'\nFase 1 selesai! -> {PHASE1_PATH}')
else:
    print(f'Fase 1 sudah selesai ({MAX_SAMPLES} sampel).')


Memulai Fase 1: 500 sampel (Hybrid + Few-Shot Multi-Query QR).
Memproses 500 sampel tersisa...

  [ 10/500] Akurasi: 50.0% | avg rewrites: 3.0 | pred=yes, gt=yes | ETA 35.6 mnt
  [ 20/500] Akurasi: 60.0% | avg rewrites: 3.0 | pred=yes, gt=yes | ETA 35.7 mnt
  [ 30/500] Akurasi: 63.3% | avg rewrites: 3.0 | pred=yes, gt=yes | ETA 35.3 mnt
  [ 40/500] Akurasi: 62.5% | avg rewrites: 2.9 | pred=no, gt=no | ETA 33.2 mnt
  [ 50/500] Akurasi: 64.0% | avg rewrites: 2.9 | pred=no, gt=no | ETA 32.1 mnt
  [ 60/500] Akurasi: 60.0% | avg rewrites: 2.9 | pred=yes, gt=yes | ETA 32.0 mnt
  [ 70/500] Akurasi: 64.3% | avg rewrites: 2.9 | pred=yes, gt=yes | ETA 31.4 mnt
  [ 80/500] Akurasi: 62.5% | avg rewrites: 2.9 | pred=yes, gt=yes | ETA 30.2 mnt
  [ 90/500] Akurasi: 62.2% | avg rewrites: 2.9 | pred=no, gt=maybe | ETA 29.4 mnt
  [100/500] Akurasi: 63.0% | avg rewrites: 2.9 | pred=yes, gt=yes | ETA 28.5 mnt
  [110/500] Akurasi: 65.5% | avg rewrites: 2.9 | pred=yes, gt=yes | ETA 28.0 mnt
  [120/500] Akur

## Analisis Phase 1

In [14]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']

n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
by_label  = {}
for r in results_p1:
    gt = r['ground_truth']
    by_label.setdefault(gt, {'total': 0, 'correct': 0})
    by_label[gt]['total']   += 1
    by_label[gt]['correct'] += int(r['is_correct'])

# Statistik multi-query
n_rewrites = [r.get('num_rewrites', 1) for r in results_p1]
avg_rw  = sum(n_rewrites) / n
max_rw  = max(n_rewrites)
min_rw  = min(n_rewrites)
dist_rw = {k: n_rewrites.count(k) for k in sorted(set(n_rewrites))}

print(f'Hasil Phase 1 (n={n}, Few-Shot Multi-Query QR):')
print(f'  Akurasi total       : {n_correct/n:.1%} ({n_correct}/{n})')
print()
print(f'Statistik jumlah rewritten queries per sampel:')
print(f'  Rata-rata           : {avg_rw:.2f}')
print(f'  Min / Max           : {min_rw} / {max_rw}')
print(f'  Distribusi          : {dist_rw}')
print()
print('Per-label breakdown:')
for label in ['yes', 'no', 'maybe']:
    d = by_label.get(label, {'total': 0, 'correct': 0})
    if d['total']:
        print(f'  {label:5s}: {d["correct"]:3d}/{d["total"]:3d} ({d["correct"]/d["total"]:.1%})')

print('\nContoh 3 sampel pertama dengan rewritten queries:')
for r in results_p1[:3]:
    print(f'  [{r["idx"]}] {r["question"][:60]}...')
    for j, rq in enumerate(r.get('rewritten_queries', []), 1):
        print(f'        rw{j}: {rq[:80]}')
    print(f'        gt={r["ground_truth"]} | pred={r["predicted_label"]} | correct={r["is_correct"]}')


ANALISIS PHASE 1 - 500 sampel (qr_openai)
Label Accuracy    : 354/500 = 70.8%
Hallucination Rate: 29.2%

Per-label accuracy:
    yes: 236/275 = 85.8%
     no: 112/159 = 70.4%
  maybe: 6/66 = 9.1%

Distribusi prediksi:
    yes: 318 (64%)
     no: 157 (31%)
  maybe: 25 (5%)

Rata-rata skor BM25  : 27.4888
Rata-rata skor Dense : 0.5557
Rata-rata skor RRF   : 0.0303


## Phase 2 — Custom Evaluator 4 Metrik (500 Sampel)

Estimasi waktu: ~20-30 menit.

In [15]:
MAX_CUSTOM_SAMPLES = 500
REQUIRED_METRICS = ['faithfulness', 'context_recall', 'answer_relevancy', 'context_precision']

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_custom = json.load(f)['results'][:MAX_CUSTOM_SAMPLES]

if PHASE2_CUSTOM_PATH.exists():
    with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
        p2_custom = json.load(f)['results']
    done_custom = {r['idx'] for r in p2_custom if all(m in r for m in REQUIRED_METRICS)}
    needs_upgrade = [r for r in p2_custom if not all(m in r for m in REQUIRED_METRICS)]
    print(f'Resume: {len(done_custom)}/{MAX_CUSTOM_SAMPLES} selesai dengan 4 metrik.')
    if needs_upgrade:
        print(f'Perlu upgrade: {len(needs_upgrade)} sampel.')
else:
    p2_custom, done_custom, needs_upgrade = [], set(), []
    print(f'Mulai: {MAX_CUSTOM_SAMPLES} sampel (custom zero-NaN, 4 metrik).')

# Tahap 1: Upgrade
if needs_upgrade:
    print(f'\nTahap 1: Upgrade {len(needs_upgrade)} sampel...')
    t_up = time.time()
    p1_lookup = {r['idx']: r for r in p1_custom}
    for i, r in enumerate(needs_upgrade):
        src = p1_lookup[r['idx']]
        if 'answer_relevancy' not in r:
            r['answer_relevancy'] = compute_answer_relevancy(src['question'], src['answer'])
        if 'context_precision' not in r:
            r['context_precision'] = compute_context_precision(src['question'], src['contexts'], src['reference'])
        done_custom.add(r['idx'])
        if (i + 1) % 5 == 0 or i == len(needs_upgrade) - 1:
            with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
                json.dump({
                    'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                    'max_samples': MAX_CUSTOM_SAMPLES,
                    'metrics': REQUIRED_METRICS,
                    'evaluator': 'custom_zero_nan_4metrics',
                    'results': p2_custom
                }, f, indent=2, ensure_ascii=False)
            done  = i + 1
            eta   = (time.time()-t_up)/done*(len(needs_upgrade)-done)/60 if done < len(needs_upgrade) else 0
            print(f'  upgrade [{done:3d}/{len(needs_upgrade)}] | ETA {eta:.1f} mnt')

# Tahap 2: Evaluasi sampel baru
remaining = [r for r in p1_custom if r['idx'] not in done_custom]
print(f'\nTahap 2: Evaluasi {len(remaining)} sampel baru...\n')

t0 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_custom(r['question'], r['answer'], r['contexts'], r['reference'])
    p2_custom.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
            json.dump({
                'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                'max_samples': MAX_CUSTOM_SAMPLES,
                'metrics': REQUIRED_METRICS,
                'evaluator': 'custom_zero_nan_4metrics',
                'results': p2_custom
            }, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t0)/done*(total-done)/60 if done < total else 0
        avg_f  = sum(x['faithfulness']      for x in p2_custom) / len(p2_custom)
        avg_cr = sum(x['context_recall']    for x in p2_custom) / len(p2_custom)
        avg_ar = sum(x['answer_relevancy']  for x in p2_custom) / len(p2_custom)
        avg_cp = sum(x['context_precision'] for x in p2_custom) / len(p2_custom)
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'f={scores["faithfulness"]:.2f} cr={scores["context_recall"]:.2f} '
              f'ar={scores["answer_relevancy"]:.2f} cp={scores["context_precision"]:.2f} | '
              f'avg: f={avg_f:.3f} cr={avg_cr:.3f} ar={avg_ar:.3f} cp={avg_cp:.3f} | ETA {eta:.1f}m')

print(f'\nSelesai! -> {PHASE2_CUSTOM_PATH}')

Mulai: 500 sampel (custom zero-NaN, 4 metrik).

Tahap 2: Evaluasi 500 sampel baru...

  [  5/500] idx=4 | f=0.67 cr=0.83 ar=1.00 cp=0.50 | avg: f=0.733 cr=0.650 ar=1.000 cp=0.683 | ETA 85.0m
  [ 10/500] idx=9 | f=0.50 cr=0.67 ar=1.00 cp=0.00 | avg: f=0.783 cr=0.658 ar=0.967 cp=0.625 | ETA 75.5m
  [ 15/500] idx=14 | f=1.00 cr=1.00 ar=1.00 cp=0.92 | avg: f=0.833 cr=0.706 ar=0.978 cp=0.606 | ETA 75.2m
  [ 20/500] idx=19 | f=1.00 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.858 cr=0.746 ar=0.983 cp=0.654 | ETA 73.3m
  [ 25/500] idx=24 | f=1.00 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.860 cr=0.797 ar=0.987 cp=0.690 | ETA 92.8m
  [ 30/500] idx=29 | f=1.00 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.872 cr=0.797 ar=0.989 cp=0.689 | ETA 86.8m
  [ 35/500] idx=34 | f=1.00 cr=1.00 ar=1.00 cp=0.83 | avg: f=0.876 cr=0.826 ar=0.990 cp=0.729 | ETA 82.0m
  [ 40/500] idx=39 | f=1.00 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.867 cr=0.835 ar=0.983 cp=0.718 | ETA 80.5m
  [ 45/500] idx=44 | f=1.00 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.841 c

## Summary — Hasil Akhir

In [16]:
with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
    p2 = json.load(f)['results']

n      = len(p2)
acc    = sum(r['is_correct']        for r in p2) / n
avg_f  = sum(r['faithfulness']      for r in p2) / n
avg_cr = sum(r['context_recall']    for r in p2) / n
avg_ar = sum(r['answer_relevancy']  for r in p2) / n
avg_cp = sum(r['context_precision'] for r in p2) / n

print('=' * 65)
print(f'  {CONFIG_NAME.upper()} (Hybrid OpenAI) - {n} sampel')
print(f'  LLM: {LLM_MODEL}')
print('=' * 65)
print(f'  Label Accuracy     : {acc:.1%}')
print(f'  Hallucination Rate : {1-acc:.1%}')
print(f'  Faithfulness       : {avg_f:.4f}')
print(f'  Context Recall     : {avg_cr:.4f}')
print(f'  Answer Relevancy   : {avg_ar:.4f}')
print(f'  Context Precision  : {avg_cp:.4f}')
print(f'  NaN count          : 0')
print('=' * 65)

print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = [r for r in p2 if r['ground_truth'] == lbl]
    if sub:
        lbl_acc = sum(r['is_correct'] for r in sub) / len(sub)
        print(f'  {lbl:>5}: {sum(r["is_correct"] for r in sub)}/{len(sub)} = {lbl_acc:.1%}')

# Perbandingan dengan konfigurasi OpenAI sebelumnya
print('\nPerbandingan dengan konfigurasi OpenAI sebelumnya:')
for prev_config, prev_path in [
    ('Baseline OpenAI', '../results/baseline_openai_phase2_custom.json'),
    ('QR OpenAI',       '../results/qr_openai_phase2_custom.json'),
    ('CR OpenAI',       '../results/cr_openai_phase2_custom.json'),
]:
    try:
        with open(prev_path, 'r', encoding='utf-8') as f:
            prev = json.load(f)['results']
        p_acc = sum(r['is_correct'] for r in prev) / len(prev)
        p_f   = sum(r['faithfulness'] for r in prev) / len(prev)
        delta = (acc - p_acc) * 100
        print(f'  {prev_config:<20}: acc={p_acc:.1%} faith={p_f:.4f} | delta acc = {delta:+.1f}%')
    except FileNotFoundError:
        print(f'  {prev_config:<20}: file tidak ditemukan')

print(f'\nBaris tabel skripsi:')
print(f'  | Hybrid OpenAI | {acc:.3f} | {1-acc:.3f} | '
      f'{avg_f:.3f} | {avg_cr:.3f} | {avg_ar:.3f} | {avg_cp:.3f} |')

  QR_OPENAI (Hybrid OpenAI) - 500 sampel
  LLM: gpt-4.1-mini
  Label Accuracy     : 70.8%
  Hallucination Rate : 29.2%
  Faithfulness       : 0.8967
  Context Recall     : 0.8234
  Answer Relevancy   : 0.9811
  Context Precision  : 0.7032
  NaN count          : 0

Per-label accuracy:
    yes: 236/275 = 85.8%
     no: 112/159 = 70.4%
  maybe: 6/66 = 9.1%

Perbandingan dengan konfigurasi OpenAI sebelumnya:
  Baseline OpenAI     : acc=69.2% faith=0.8930 | delta acc = +1.6%
  QR OpenAI           : acc=70.8% faith=0.8967 | delta acc = +0.0%
  CR OpenAI           : acc=69.8% faith=0.8837 | delta acc = +1.0%

Baris tabel skripsi:
  | Hybrid OpenAI | 0.708 | 0.292 | 0.897 | 0.823 | 0.981 | 0.703 |
